
# 03 — Feature Engineering (Customers)

**Purpose:** Create clean, interpretable features that serve both goals  
- **Goal 1:** Predict campaign **Response** (binary)  
- **Goal 3:** Build **personas/clusters**

**Inputs**
- `cleaned_marketing_campaign.csv` (cleaned by DE)

**Outputs**
- `customers_featured.csv` (feature-enriched table)

> Tip: Run this notebook top-to-bottom. The next notebook (04 EDA) expects `customers_featured.csv` to exist.


In [40]:

# --- Imports & paths
import os
import numpy as np
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Set paths (edit if needed)
DATA_IN  = "cleaned_marketing_campaign.csv"
DATA_OUT = "customers_featured.csv"

# Display options
pd.set_option("display.max_columns", 200)


In [41]:

# --- Load cleaned data
df = pd.read_csv(DATA_IN)
print("Loaded:", DATA_IN, "→ shape:", df.shape)
display(df.head(3))


Loaded: cleaned_marketing_campaign.csv → shape: (2216, 27)


,Education,Has_Partner,Income,Recency,Wines,Fruits,Meat,Fish,Sweets,Gold,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Response,Age,Total_Spent,Spending_Ratio,Children_Home,Customer_Tenure_Days
0,Graduation,0,58138,58,635,88,546,172,88,88,3,8,10,4,7,0,0,0,0,0,0,1,57,1617,0.027813,0,663
1,Graduation,0,46344,38,11,1,6,2,1,6,2,1,1,2,5,0,0,0,0,0,0,0,60,27,0.000583,2,113
2,Graduation,1,71613,26,426,49,127,111,21,42,1,8,2,10,4,0,0,0,0,0,0,0,49,776,0.010836,0,312



## Safety helpers


In [42]:

def safe_div(numer, denom):
    denom = np.where(denom==0, np.nan, denom)
    return numer / denom



## Feature set
We engineer features across 6 areas:

1) **Family & per-capita**  
2) **Category mix & preference**  
3) **Channel & engagement**  
4) **Campaign history**  
5) **Recency & RFM**  
6) **Transforms & simple interactions**


### 1) Family & per-capita

In [43]:

# Assumes these columns exist in the cleaned data:
# 'Children_Home', 'Has_Partner', 'Income', 'Total_Spent', 'Age', 'Customer_Tenure_Days', 'Recency'
req_cols = ["Children_Home","Has_Partner","Income","Total_Spent","Age","Customer_Tenure_Days","Recency"]
missing = [c for c in req_cols if c not in df.columns]
if missing:
    print("WARNING: Missing expected columns:", missing)

df["Family_Size"] = df["Children_Home"] + (1 + df["Has_Partner"])  # adults = 1 or 2
df["Income_per_Capita"] = safe_div(df["Income"], df["Family_Size"])
df["Spend_per_Capita"] = safe_div(df["Total_Spent"], df["Family_Size"])
df["Dependency_Ratio"] = safe_div(df["Children_Home"], df["Family_Size"])

display(df[["Family_Size","Income_per_Capita","Spend_per_Capita","Dependency_Ratio"]].head(5))


,Family_Size,Income_per_Capita,Spend_per_Capita,Dependency_Ratio
0,1,58138.0,1617.000000,0.000000
1,3,15448.0,9.000000,0.666667
2,2,35806.5,388.000000,0.000000
3,3,8882.0,17.666667,0.333333
4,3,19431.0,140.666667,0.333333


### 2) Category mix & preference

In [44]:
# 2) Category mix & preference  — robust to zero spend
cat_cols = ["Wines","Fruits","Meat","Fish","Sweets","Gold"]
for c in cat_cols:
    if c not in df.columns:
        print("WARNING: spend category missing:", c)

# shares using safe_div; NaN when Total_Spent==0
total_spent_nonzero = df["Total_Spent"].replace(0, np.nan)
for c in cat_cols:
    df[f"{c}_Share"] = safe_div(df[c], total_spent_nonzero)

# >>> NEW: set shares to 0 when Total_Spent == 0 (instead of NaN)
share_cols = [f"{c}_Share" for c in cat_cols]
df.loc[df["Total_Spent"] == 0, share_cols] = 0.0

df["Num_Categories_Active"] = (df[cat_cols] > 0).sum(axis=1)

# >>> NEW: concentration on filled shares (no NaNs)
df["Category_Concentration"] = df[share_cols].pow(2).sum(axis=1)

df["Premium_Tilt"] = safe_div(df["Gold"] + df["Wines"], total_spent_nonzero)

display(df[share_cols + ["Num_Categories_Active","Category_Concentration","Premium_Tilt"]].head(5))


,Wines_Share,Fruits_Share,Meat_Share,Fish_Share,Sweets_Share,Gold_Share,Num_Categories_Active,Category_Concentration,Premium_Tilt
0,0.392703,0.054422,0.337662,0.106370,0.054422,0.054422,6,0.288431,0.447124
1,0.407407,0.037037,0.222222,0.074074,0.037037,0.222222,6,0.272977,0.629630
2,0.548969,0.063144,0.163660,0.143041,0.027062,0.054124,6,0.356261,0.603093
3,0.207547,0.075472,0.377358,0.188679,0.056604,0.094340,6,0.238875,0.301887
4,0.409953,0.101896,0.279621,0.109005,0.063981,0.035545,6,0.273871,0.445498


### 3) Channel & engagement

In [45]:

# Use channel purchases: web, catalog, store (deals is orthogonal)
purchase_cols = ["NumWebPurchases","NumCatalogPurchases","NumStorePurchases"]
missing_p = [c for c in purchase_cols if c not in df.columns]
if missing_p:
    print("WARNING: Missing purchase columns:", missing_p)

df["Total_Purchases"] = df["NumWebPurchases"] + df["NumCatalogPurchases"] + df["NumStorePurchases"]
df["Avg_Spend_per_Purchase"] = safe_div(df["Total_Spent"], df["Total_Purchases"])

df["Web_Share"]     = safe_div(df["NumWebPurchases"], df["Total_Purchases"])
df["Catalog_Share"] = safe_div(df["NumCatalogPurchases"], df["Total_Purchases"])
df["Store_Share"]   = safe_div(df["NumStorePurchases"], df["Total_Purchases"])

df["Deals_Share"]   = safe_div(df["NumDealsPurchases"], df["Total_Purchases"]) if "NumDealsPurchases" in df.columns else np.nan
df.drop(columns=["Web_Conversion_Approx"], errors="ignore", inplace=True)

display(df[["Total_Purchases","Avg_Spend_per_Purchase","Web_Share","Catalog_Share","Store_Share","Deals_Share"]].head(5))


,Total_Purchases,Avg_Spend_per_Purchase,Web_Share,Catalog_Share,Store_Share,Deals_Share
0,22,73.500000,0.363636,0.454545,0.181818,0.136364
1,4,6.750000,0.250000,0.250000,0.500000,0.500000
2,20,38.800000,0.400000,0.100000,0.500000,0.050000
3,6,8.833333,0.333333,0.000000,0.666667,0.333333
4,14,30.142857,0.357143,0.214286,0.428571,0.357143


### 4) Campaign history

In [46]:

cmp_cols = [c for c in ["AcceptedCmp1","AcceptedCmp2","AcceptedCmp3","AcceptedCmp4","AcceptedCmp5"] if c in df.columns]
df["Past_Accepted"] = df[cmp_cols].sum(axis=1) if cmp_cols else 0
df["Ever_Accepted_Past"] = (df["Past_Accepted"] > 0).astype(int)
df["Campaign_Accept_Rate"] = safe_div(df["Past_Accepted"], len(cmp_cols) if cmp_cols else np.nan)

display(df[["Past_Accepted","Ever_Accepted_Past","Campaign_Accept_Rate"]].head(5))


,Past_Accepted,Ever_Accepted_Past,Campaign_Accept_Rate
0,0,0,0.0
1,0,0,0.0
2,0,0,0.0
3,0,0,0.0
4,0,0,0.0


### 5) Recency & RFM

In [47]:

# Tenure months
df["Tenure_Months"] = np.round(safe_div(df["Customer_Tenure_Days"], 30)).fillna(0)
df["Frequency_pm"] = safe_div(df["Total_Purchases"], df["Tenure_Months"].replace(0, np.nan))

df["Monetary"] = df["Total_Spent"]
df["log_Monetary"] = np.log1p(df["Monetary"])

# Recency bins
df["Recency_Bin"] = pd.cut(
    df["Recency"],
    bins=[-np.inf,30,90,np.inf],
    labels=["0-30 (Hot)","31-90 (Warm)",">90 (Cold)"]
)

# RFM quantile scores
def quantile_score(s, high_is_good=True, q=[0, .2, .4, .6, .8, 1.0]):
    bins = s.quantile(q).values
    bins = np.unique(bins)
    if len(bins) < 3:
        return pd.Series(3, index=s.index)  # fallback constant
    labels = list(range(1, len(bins)))
    scores = pd.cut(s, bins=bins, labels=labels, include_lowest=True, duplicates='drop')
    scores = scores.astype(float)
    if not high_is_good:
        max_score = float(np.nanmax(scores))
        scores = max_score + 1 - scores
    return scores

df["R_score"] = quantile_score(df["Recency"], high_is_good=False)
df["F_score"] = quantile_score(df["Frequency_pm"], high_is_good=True)
df["M_score"] = quantile_score(df["Monetary"], high_is_good=True)
df["RFM_Total"] = df[["R_score","F_score","M_score"]].sum(axis=1)

display(df[["Recency","Recency_Bin","Tenure_Months","Frequency_pm","Monetary","R_score","F_score","M_score","RFM_Total"]].head(5))


,Recency,Recency_Bin,Tenure_Months,Frequency_pm,Monetary,R_score,F_score,M_score,RFM_Total
0,58,31-90 (Warm),22.0,1.0,1617,3.0,3.0,5.0,11.0
1,38,31-90 (Warm),4.0,1.0,27,4.0,3.0,1.0,8.0
2,26,0-30 (Hot),10.0,2.0,776,4.0,4.0,4.0,12.0
3,26,0-30 (Hot),5.0,1.2,53,4.0,3.0,1.0,8.0
4,94,>90 (Cold),5.0,2.8,422,1.0,5.0,3.0,9.0


### 6) Transforms & simple interactions + bins

In [48]:

df["log_Income"] = np.log1p(df["Income"])
df["log_Total_Spent"] = np.log1p(df["Total_Spent"])

df["Income_x_HasPartner"] = df["Income"] * df["Has_Partner"]
df["Age_x_WebShare"] = df["Age"] * df["Web_Share"]

df["Age_Bin"] = pd.cut(df["Age"], bins=[-np.inf, 29, 44, 59, np.inf], labels=["18-29","30-44","45-59","60+"])
df["Income_q"] = pd.qcut(df["Income"], q=5, labels=False, duplicates="drop")

new_cols = [c for c in df.columns if c not in req_cols and c not in cat_cols]
print("Engineered columns (sample):", new_cols[:15], "... total new cols:", len(new_cols))


Engineered columns (sample): ['Education', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1', 'AcceptedCmp2', 'Complain', 'Response', 'Spending_Ratio', 'Family_Size'] ... total new cols: 51


## Save featured table

In [49]:

df.to_csv(DATA_OUT, index=False)
print("Saved:", DATA_OUT, "→ shape:", df.shape)
display(df.head(3))


Saved: customers_featured.csv → shape: (2216, 64)


,Education,Has_Partner,Income,Recency,Wines,Fruits,Meat,Fish,Sweets,Gold,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Response,Age,Total_Spent,Spending_Ratio,Children_Home,Customer_Tenure_Days,Family_Size,Income_per_Capita,Spend_per_Capita,Dependency_Ratio,Wines_Share,Fruits_Share,Meat_Share,Fish_Share,Sweets_Share,Gold_Share,Num_Categories_Active,Category_Concentration,Premium_Tilt,Total_Purchases,Avg_Spend_per_Purchase,Web_Share,Catalog_Share,Store_Share,Deals_Share,Past_Accepted,Ever_Accepted_Past,Campaign_Accept_Rate,Tenure_Months,Frequency_pm,Monetary,log_Monetary,Recency_Bin,R_score,F_score,M_score,RFM_Total,log_Income,log_Total_Spent,Income_x_HasPartner,Age_x_WebShare,Age_Bin,Income_q
0,Graduation,0,58138,58,635,88,546,172,88,88,3,8,10,4,7,0,0,0,0,0,0,1,57,1617,0.027813,0,663,1,58138.0,1617.0,0.000000,0.392703,0.054422,0.337662,0.106370,0.054422,0.054422,6,0.288431,0.447124,22,73.50,0.363636,0.454545,0.181818,0.136364,0,0,0.0,22.0,1.0,1617,7.388946,31-90 (Warm),3.0,3.0,5.0,11.0,10.970592,7.388946,0,20.727273,45-59,2
1,Graduation,0,46344,38,11,1,6,2,1,6,2,1,1,2,5,0,0,0,0,0,0,0,60,27,0.000583,2,113,3,15448.0,9.0,0.666667,0.407407,0.037037,0.222222,0.074074,0.037037,0.222222,6,0.272977,0.629630,4,6.75,0.250000,0.250000,0.500000,0.500000,0,0,0.0,4.0,1.0,27,3.332205,31-90 (Warm),4.0,3.0,1.0,8.0,10.743869,3.332205,0,15.000000,60+,2
2,Graduation,1,71613,26,426,49,127,111,21,42,1,8,2,10,4,0,0,0,0,0,0,0,49,776,0.010836,0,312,2,35806.5,388.0,0.000000,0.548969,0.063144,0.163660,0.143041,0.027062,0.054124,6,0.356261,0.603093,20,38.80,0.400000,0.100000,0.500000,0.050000,0,0,0.0,10.0,2.0,776,6.655440,0-30 (Hot),4.0,4.0,4.0,12.0,11.179046,6.655440,71613,19.600000,45-59,3
